# Calculate NDVI for Sentinel-2 Image

This notebook calculates the Normalized Difference Vegetation Index (NDVI) for a Sentinel-2 image.

## What is NDVI?

NDVI (Normalized Difference Vegetation Index) is a vegetation index that uses the difference between near-infrared (NIR) and red light reflectance to assess vegetation health and density. Values range from -1 to 1:

- **Values close to 1**: Dense, healthy vegetation
- **Values around 0**: Bare soil or sparse vegetation
- **Values close to -1**: Water or other non-vegetated surfaces

## Parameters

This notebook has been automatically configured with the following parameters:

- **Collection**: {{STAC_COLLECTION_NAME}}
- **STAC Item**: {{STAC_ITEM_LINK}}
- **AOI (Area of Interest)**: {{AOI}} *(optional - if not provided, a 1000x1000 pixel sample will be used)*

## Workflow

1. Load the STAC item
2. Access the red (B04) and NIR (B08) bands
3. Calculate NDVI using the formula: (NIR - Red) / (NIR + Red)
4. Visualize the results

## Step 1: Import Required Libraries

In [ ]:
import pystac
import rasterio
from rasterio.windows import Window
from rasterio.mask import mask
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import json
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

## Step 2: Determine Area of Interest (AOI)

Determine the area to process. If an AOI is provided, clip to that geometry. Otherwise, extract a 1000x1000 pixel sample from the center of the image.

In [ ]:
# AOI parameter - must be GeoJSON geometry (Polygon, MultiPolygon, etc.)# Using triple quotes to safely handle JSON strings with quotesaoi_param = """{{AOI}}""".strip()# Default window size if no AOI providedDEFAULT_WINDOW_SIZE = 1000  # pixelsaoi_geometry = Noneclip_window = Noneuse_windowed_read = False# Check if AOI is providedif aoi_param and aoi_param.lower() not in ["", "none", "null"]:    try:        # Parse as JSON        aoi_data = json.loads(aoi_param)        # Extract geometry from GeoJSON structure        if aoi_data.get("type") == "FeatureCollection":            # Extract first geometry from FeatureCollection            if aoi_data.get("features") and len(aoi_data["features"]) > 0:                aoi_geometry = aoi_data["features"][0].get("geometry")        elif aoi_data.get("type") == "Feature":            # Extract geometry from Feature            aoi_geometry = aoi_data.get("geometry")        elif aoi_data.get("type") in ["Polygon", "MultiPolygon", "Point", "LineString"]:            # Direct geometry object            aoi_geometry = aoi_data        else:            raise ValueError(f"Unsupported GeoJSON type: {aoi_data.get('type')}")        # Validate geometry was extracted        if aoi_geometry and aoi_geometry.get("type"):            print(f"AOI provided: {aoi_geometry['type']} geometry")            print("Will clip raster to AOI geometry")        else:            raise ValueError("Could not extract geometry from GeoJSON")    except (json.JSONDecodeError, ValueError, KeyError) as e:        print(f"Warning: Could not parse AOI as GeoJSON: {e}")        print("Falling back to default 1000x1000 pixel window")        aoi_geometry = Noneelse:    print("No AOI provided, using default 1000x1000 pixel window")# Set up windowed read if no AOI geometryif aoi_geometry is None:    use_windowed_read = True    print(        f"Will extract {DEFAULT_WINDOW_SIZE}x{DEFAULT_WINDOW_SIZE} pixel window from center"    )

In [ ]:
# STAC item URL - automatically populated from selected dataset
stac_item_url = "{{STAC_ITEM_LINK}}"
stac_collection_name = "{{STAC_COLLECTION_NAME}}"

try:
    # Load the STAC item
    item = pystac.Item.from_file(stac_item_url)

    print(f"Successfully loaded STAC item: {item.id}")
    print(f"Collection: {stac_collection_name}")
    print(f"Date: {item.datetime}")
    print(f"Geometry: {item.geometry}")

except Exception as e:
    print(f"Error loading STAC item: {e}")
    raise

## Step 3: Access Sentinel-2 Bands

For NDVI calculation, we need:
- **Red band (B04)**: Wavelength ~665 nm
- **NIR band (B08)**: Wavelength ~842 nm

In [ ]:
try:
    # Find the red band (B04) and NIR band (B08) - either as separate assets or inside a multi-band COG
    red_band_asset = None
    nir_band_asset = None
    cog_asset = None  # Multi-band COG containing all bands (e.g. CEDA Sentinel-2 ARD)
    red_band_index = 4  # 1-based band index for B04 in standard S2 order
    nir_band_index = 8  # 1-based band index for B08 in standard S2 order

    # 1) Look for separate per-band assets first
    for asset_key, asset in item.assets.items():
        if "B04" in asset_key.upper() or (
            "red" in asset_key.lower() and "visual" not in asset_key.lower()
        ):
            red_band_asset = asset
            print(f"Found red band (B04): {asset_key}")
        if "B08" in asset_key.upper() or (
            "nir" in asset_key.lower() and "B08" in asset_key.upper()
        ):
            nir_band_asset = asset
            print(f"Found NIR band (B08): {asset_key}")

    # 2) If no per-band assets, look for a single multi-band COG (e.g. 'cog', 'data', 'image')
    if red_band_asset is None or nir_band_asset is None:
        for asset_key in ["cog", "data", "image", "reflectance", "bands"]:
            if asset_key in item.assets:
                cog_asset = item.assets[asset_key]
                # Try to get band indices from STAC eo:bands if present
                extra = getattr(cog_asset, "extra", {}) or {}
                eo_bands = extra.get("eo:bands", item.properties.get("eo:bands", []))
                if eo_bands:
                    for i, b in enumerate(eo_bands):
                        name = (b.get("name") or b.get("common_name") or "").upper()
                        if "B04" in name or b.get("common_name") == "red":
                            red_band_index = i + 1
                        if "B08" in name or b.get("common_name") == "nir":
                            nir_band_index = i + 1
                print(
                    f"Using multi-band asset '{asset_key}' (B04=band {red_band_index}, B08=band {nir_band_index})"
                )
                break
        if cog_asset is None:
            raise ValueError(
                "Could not find red (B04) or NIR (B08) bands. "
                "This STAC item has no per-band assets and no multi-band 'cog'/'data' asset."
            )

    if red_band_asset is not None and nir_band_asset is not None:
        print("\nBand assets found successfully!")
        print(f"Red band href: {red_band_asset.href}")
        print(f"NIR band href: {nir_band_asset.href}")
    else:
        print(f"\nMulti-band COG href: {cog_asset.href}")

except Exception as e:
    print(f"Error accessing bands: {e}")
    print("\nAvailable assets:")
    for asset_key in item.assets.keys():
        print(f"  - {asset_key}")
    raise

## Step 4: Read Band Data

Read the red and NIR band data into numpy arrays. If an AOI was provided, the data will be clipped to that geometry. Otherwise, a 1000x1000 pixel window will be extracted from the center.

In [ ]:
try:    # Ensure AOI variables are initialized (in case AOI cell was not executed)    try:        _ = aoi_geometry    except NameError:        aoi_geometry = None        clip_window = None        use_windowed_read = False    if cog_asset is not None:        # Read B04 and B08 from multi-band COG (1-based band indices)        with rasterio.open(cog_asset.href) as src:            if src.count < max(red_band_index, nir_band_index):                raise ValueError(                    f"COG has {src.count} bands; need at least band {max(red_band_index, nir_band_index)}. "                    "Check band order (e.g. B01,B02,B03,B04,...)."                )            # Determine clip window or use default window            if aoi_geometry is not None:                # Clip to AOI geometry                red_data, red_transform = mask(                    src, [aoi_geometry], crop=True, indexes=[red_band_index]                )                nir_data, nir_transform = mask(                    src, [aoi_geometry], crop=True, indexes=[nir_band_index]                )                red_data = red_data[0]  # Remove band dimension                nir_data = nir_data[0]                # Update profile with clipped bounds                red_profile = src.profile.copy()                red_profile.update(                    {                        "height": red_data.shape[0],                        "width": red_data.shape[1],                        "transform": red_transform,                        "count": 1,                    }                )                print(f"Clipped to AOI geometry: {red_data.shape}")            elif use_windowed_read:                # Extract center 1000x1000 pixel window                height, width = src.height, src.width                if height < DEFAULT_WINDOW_SIZE or width < DEFAULT_WINDOW_SIZE:                    print(                        f"Image is smaller than {DEFAULT_WINDOW_SIZE}x{DEFAULT_WINDOW_SIZE}, using full image"                    )                    clip_window = None                else:                    row_off = (height - DEFAULT_WINDOW_SIZE) // 2                    col_off = (width - DEFAULT_WINDOW_SIZE) // 2                    clip_window = Window(                        col_off, row_off, DEFAULT_WINDOW_SIZE, DEFAULT_WINDOW_SIZE                    )                    print(                        f"Extracting {DEFAULT_WINDOW_SIZE}x{DEFAULT_WINDOW_SIZE} pixel window from center"                    )                red_data = src.read(red_band_index, window=clip_window)                nir_data = src.read(nir_band_index, window=clip_window)                red_profile = src.profile.copy()                if clip_window:                    red_profile.update(                        {                            "height": clip_window.height,                            "width": clip_window.width,                            "transform": rasterio.windows.transform(                                clip_window, src.transform                            ),                            "count": 1,                        }                    )                else:                    red_profile.update(count=1)            else:                # Read full image                red_data = src.read(red_band_index)                nir_data = src.read(nir_band_index)                red_profile = src.profile.copy()                red_profile.update(count=1)            red_crs = src.crs            print(                f"Read from multi-band COG: band {red_band_index} (red), band {nir_band_index} (NIR)"            )    else:        # Read from separate band assets        with rasterio.open(red_band_asset.href) as red_src:            if aoi_geometry is not None:                red_data, red_transform = mask(red_src, [aoi_geometry], crop=True)                red_data = red_data[0]                red_profile = red_src.profile.copy()                red_profile.update(                    {                        "height": red_data.shape[0],                        "width": red_data.shape[1],                        "transform": red_transform,                        "count": 1,                    }                )            elif use_windowed_read:                height, width = red_src.height, red_src.width                if height < DEFAULT_WINDOW_SIZE or width < DEFAULT_WINDOW_SIZE:                    clip_window = None                else:                    row_off = (height - DEFAULT_WINDOW_SIZE) // 2                    col_off = (width - DEFAULT_WINDOW_SIZE) // 2                    clip_window = Window(                        col_off, row_off, DEFAULT_WINDOW_SIZE, DEFAULT_WINDOW_SIZE                    )                red_data = red_src.read(1, window=clip_window)                red_profile = red_src.profile.copy()                if clip_window:                    red_profile.update(                        {                            "height": clip_window.height,                            "width": clip_window.width,                            "transform": rasterio.windows.transform(                                clip_window, red_src.transform                            ),                        }                    )            else:                red_data = red_src.read(1)                red_profile = red_src.profile.copy()            red_crs = red_src.crs        with rasterio.open(nir_band_asset.href) as nir_src:            if aoi_geometry is not None:                nir_data, nir_transform = mask(nir_src, [aoi_geometry], crop=True)                nir_data = nir_data[0]            elif use_windowed_read:                nir_data = nir_src.read(1, window=clip_window)            else:                nir_data = nir_src.read(1)            # Use same profile as red (same grid)            red_profile = nir_src.profile.copy()            if aoi_geometry is not None:                red_profile.update(                    {                        "height": nir_data.shape[0],                        "width": nir_data.shape[1],                        "transform": nir_transform,                        "count": 1,                    }                )            elif clip_window:                red_profile.update(                    {                        "height": clip_window.height,                        "width": clip_window.width,                        "transform": rasterio.windows.transform(                            clip_window, nir_src.transform                        ),                    }                )    print(f"Red band shape: {red_data.shape}, dtype: {red_data.dtype}")    print(f"NIR band shape: {nir_data.shape}, dtype: {nir_data.dtype}")    print(f"CRS: {red_crs}")    if red_data.shape != nir_data.shape:        raise ValueError(            f"Band shapes do not match: Red {red_data.shape} vs NIR {nir_data.shape}"        )    red_data = red_data.astype(np.float32)    nir_data = nir_data.astype(np.float32)    print(f"\nBand data loaded successfully! Processing {red_data.size:,} pixels")except Exception as e:    print(f"Error reading band data: {e}")    raise

## Step 5: Calculate NDVI

Calculate NDVI using the formula:

$$NDVI = \frac{NIR - Red}{NIR + Red}$$

The result will be a value between -1 and 1.

In [ ]:
try:
    # Calculate NDVI
    # Avoid division by zero by adding a small epsilon
    denominator = nir_data + red_data

    # Create mask for valid pixels (where denominator is not zero)
    valid_mask = denominator != 0

    # Initialize NDVI array with NaN for invalid pixels
    ndvi = np.full_like(red_data, np.nan, dtype=np.float32)

    # Calculate NDVI only for valid pixels
    ndvi[valid_mask] = (nir_data[valid_mask] - red_data[valid_mask]) / denominator[
        valid_mask
    ]

    # Clip values to valid NDVI range [-1, 1]
    ndvi = np.clip(ndvi, -1.0, 1.0)

    print("NDVI calculation complete!")
    print(f"NDVI shape: {ndvi.shape}")
    print(f"NDVI min: {np.nanmin(ndvi):.4f}")
    print(f"NDVI max: {np.nanmax(ndvi):.4f}")
    print(f"NDVI mean: {np.nanmean(ndvi):.4f}")
    print(f"Valid pixels: {np.sum(~np.isnan(ndvi)):,} out of {ndvi.size:,}")

except Exception as e:
    print(f"Error calculating NDVI: {e}")
    raise

## Step 6: Visualize Results

Create visualizations of the NDVI results.

In [ ]:
# Create a custom colormap for NDVI visualization
# Colors: water (blue) -> bare soil (brown) -> sparse vegetation (yellow) -> dense vegetation (green)
colors = ["#000080", "#0066CC", "#CCCCCC", "#FFFF00", "#00FF00", "#008000"]
n_bins = 256
cmap = LinearSegmentedColormap.from_list("ndvi", colors, N=n_bins)

# Create figure with subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Create visualization of red band
im1 = axes[0].imshow(red_data, cmap="Reds", vmin=0, vmax=np.percentile(red_data, 98))
axes[0].set_title("Red Band (B04)", fontsize=14, fontweight="bold")
axes[0].axis("off")
plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04, label="Reflectance")

# Create visualization of NIR band
im2 = axes[1].imshow(nir_data, cmap="YlGn", vmin=0, vmax=np.percentile(nir_data, 98))
axes[1].set_title("NIR Band (B08)", fontsize=14, fontweight="bold")
axes[1].axis("off")
plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04, label="Reflectance")

# Create visualization of NDVI
im3 = axes[2].imshow(ndvi, cmap=cmap, vmin=-1, vmax=1)
axes[2].set_title("NDVI", fontsize=14, fontweight="bold")
axes[2].axis("off")
cbar = plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04, label="NDVI")

# Add colorbar labels
cbar.set_ticks([-1, -0.5, 0, 0.3, 0.6, 1])
cbar.set_ticklabels(["Water", "Bare Soil", "Sparse", "Moderate", "Dense", "Very Dense"])

plt.suptitle(f"NDVI Analysis - {item.id}", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Visualization complete!")

## Summary

This notebook has successfully:

1. ✅ Loaded the STAC item from: `{{STAC_ITEM_LINK}}`
2. ✅ Determined area of interest (AOI or default window)
3. ✅ Accessed the red (B04) and NIR (B08) bands
4. ✅ Calculated NDVI values
5. ✅ Visualized the results

### Next Steps

You can now:
- Analyze specific areas of interest
- Export the NDVI results for further analysis
- Compare NDVI values across different dates
- Create time series analyses

### Resources

- **Collection**: {{STAC_COLLECTION_NAME}}

---

*This notebook was automatically generated from a template and configured with your selected dataset.*